In [1]:
# Pregunta: ¿cuántas horas, millas y semanas de trabajo acumuló cada conductor?
# Este caso aplica los operadores de P519 a todos los turnos de la operación de transporte.
#
# lectura ──► mapper del caso ──► shuffle ──► reducer del caso ──► escritura

import csv
from pathlib import Path

ROOT = next(
    path
    for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (path / "data").is_dir() and (path / "submission").is_dir()
)

with (ROOT / "data" / "timesheet.csv").open(encoding="utf-8", newline="") as file:
    timesheet = list(csv.DictReader(file))

len(timesheet), timesheet[:2]

(1768,
 [{'driverId': '10',
   'week': '1',
   'hours-logged': '70',
   'miles-logged': '3300'},
  {'driverId': '10',
   'week': '2',
   'hours-logged': '70',
   'miles-logged': '3300'}])

In [2]:
# Estos operadores ya se estudiaron en P519; aquí se usan para resolver el caso.

def map_pairs(records, mapper):
    pairs = []

    for record in records:
        pairs.extend(mapper(record))

    return pairs


def group_by_key(pairs):
    grouped_values = {}

    for key, value in pairs:
        grouped_values.setdefault(key, []).append(value)

    return grouped_values


def reduce_by_key(grouped_values, reducer):
    return [reducer(key, values) for key, values in grouped_values.items()]

In [3]:
# Esta regla del caso convierte cada turno en una pareja con sus medidas parciales.

def map_timesheet_record(record):
    return [
        (
            record["driverId"],
            (
                float(record["hours-logged"]),
                float(record["miles-logged"]),
                1,
            ),
        )
    ]

In [4]:
# La consulta SQL especifica el resultado que se construirá con los operadores clave–valor.
#
# SELECT
#     driverId,
#     SUM("hours-logged") AS total_hours,
#     SUM("miles-logged") AS total_miles,
#     COUNT(*) AS weeks
# FROM timesheet
# GROUP BY driverId;

mapped_pairs = map_pairs(timesheet, map_timesheet_record)

mapped_pairs[:5]

[('10', (70.0, 3300.0, 1)),
 ('10', (70.0, 3300.0, 1)),
 ('10', (60.0, 2800.0, 1)),
 ('10', (70.0, 3100.0, 1)),
 ('10', (70.0, 3200.0, 1))]

In [5]:
# Esta regla del caso resume el grupo de semanas asociado a un conductor.

def summarize_driver_metrics(driver_id, values):
    total_hours = sum(hours for hours, _, _ in values)
    total_miles = sum(miles for _, miles, _ in values)
    weeks = sum(week_count for _, _, week_count in values)

    return driver_id, {
        "total_hours": round(total_hours, 2),
        "total_miles": round(total_miles, 2),
        "weeks": weeks,
        "mean_hours": round(total_hours / weeks, 2),
    }

In [6]:
# Los operadores genéricos producen una pareja reducida por conductor.

values_by_driver = group_by_key(mapped_pairs)
driver_metric_pairs = reduce_by_key(values_by_driver, summarize_driver_metrics)
driver_metric_pairs.sort(key=lambda pair: int(pair[0]))

driver_metrics = [
    {"driverId": driver_id, **metrics}
    for driver_id, metrics in driver_metric_pairs
]

with (ROOT / "submission" / "driver_metrics.csv").open(
    "w", encoding="utf-8", newline=""
) as file:
    writer = csv.DictWriter(file, fieldnames=list(driver_metrics[0]), lineterminator="\n")
    writer.writeheader()
    writer.writerows(driver_metrics)

driver_metrics[:5]

[{'driverId': '10',
  'total_hours': 3232.0,
  'total_miles': 147150.0,
  'weeks': 52,
  'mean_hours': 62.15},
 {'driverId': '11',
  'total_hours': 3642.0,
  'total_miles': 179300.0,
  'weeks': 52,
  'mean_hours': 70.04},
 {'driverId': '12',
  'total_hours': 2639.0,
  'total_miles': 135962.0,
  'weeks': 52,
  'mean_hours': 50.75},
 {'driverId': '13',
  'total_hours': 2727.0,
  'total_miles': 134126.0,
  'weeks': 52,
  'mean_hours': 52.44},
 {'driverId': '14',
  'total_hours': 2781.0,
  'total_miles': 136624.0,
  'weeks': 52,
  'mean_hours': 53.48}]